In [2]:
# EXTRAÇÃO DE ONSETS (TEMPOS EXATOS DOS ESTÍMULOS)

import os
import pandas as pd
import mne
import warnings

# Silenciando avisos para terminal limpo
warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')

print("\n" + "="*80)
print("Iniciando Mineração de Triggers (FF, FN, FR) no arquivo bruto...")
print("="*80)

# 1. Caminhos e Parâmetros
DIR_RAW = '../data/raw/'
DIR_RESULTS = '../results/'

# Lista de pacientes com falha conhecida (herdada do seu setup)
ARQUIVOS_EXCLUIDOS = ['control_14-sem marcacoes', 'TEA_M_22_1']
ESTIMULOS_ALVO = ['FF', 'FN', 'FR']

# Criar pasta de resultados se não existir
os.makedirs(DIR_RESULTS, exist_ok=True)

# 2. Varredura
arquivos_edf = sorted([f for f in os.listdir(DIR_RAW) if f.lower().endswith('.edf')])
dados_onsets = []
pacientes_processados = 0

for arquivo in arquivos_edf:
    paciente_id = arquivo.replace('.edf', '')
    
    # Pula pacientes excluídos
    if paciente_id in ARQUIVOS_EXCLUIDOS:
        continue
        
    caminho_completo = os.path.join(DIR_RAW, arquivo)
    
    try:
        # preload=False garante que só os metadados/eventos sejam lidos (super rápido)
        raw = mne.io.read_raw_edf(caminho_completo, preload=False, encoding='latin1', verbose=False)
        
        contador_estimulos = 0
        
        # Itera sobre todas as anotações do exame
        for ann in raw.annotations:
            descricao = ann['description'].strip().upper()
            
            # Se a descrição for exatamente FF, FN ou FR, nós extraímos
            if descricao in ESTIMULOS_ALVO:
                dados_onsets.append({
                    'Paciente_ID': paciente_id,
                    'Estimulo': descricao,
                    'Onset_Segundos': ann['onset']
                })
                contador_estimulos += 1
                
        print(f"[{paciente_id}] -> {contador_estimulos} estímulos mapeados.")
        pacientes_processados += 1
        
    except Exception as e:
        print(f"[ERRO] Falha ao processar {paciente_id}: {e}")

# 3. Consolidação e Exportação
df_onsets = pd.DataFrame(dados_onsets)

# Salva o arquivo CSV definitivo
caminho_csv = os.path.join(DIR_RESULTS, 'tabela_tempos_estimulos_faces.csv')
df_onsets.to_csv(caminho_csv, index=False)

print("\n" + "="*80)
print("EXTRAÇÃO CONCLUÍDA COM SUCESSO!")
print(f"Total de pacientes processados: {pacientes_processados}")
print(f"Total de estímulos isolados: {len(df_onsets)}")
print(f"Arquivo salvo em: {caminho_csv}")
print("="*80)

# Exibe as primeiras linhas para conferência visual imediata
display(df_onsets.head(10))


Iniciando Mineração de Triggers (FF, FN, FR) no arquivo bruto...
[TEA_G_26] -> 30 estímulos mapeados.
[TEA_G_28] -> 30 estímulos mapeados.
[TEA_G_30] -> 30 estímulos mapeados.
[TEA_G_40] -> 30 estímulos mapeados.
[TEA_L_16] -> 0 estímulos mapeados.
[TEA_L_17] -> 30 estímulos mapeados.
[TEA_L_24] -> 30 estímulos mapeados.
[TEA_L_25] -> 27 estímulos mapeados.
[TEA_L_27] -> 31 estímulos mapeados.
[TEA_L_29] -> 30 estímulos mapeados.
[TEA_L_32] -> 30 estímulos mapeados.
[TEA_L_46] -> 30 estímulos mapeados.
[TEA_M_15] -> 30 estímulos mapeados.
[TEA_M_18] -> 30 estímulos mapeados.
[TEA_M_19] -> 30 estímulos mapeados.
[TEA_M_20] -> 30 estímulos mapeados.
[TEA_M_21] -> 30 estímulos mapeados.
[TEA_M_22] -> 30 estímulos mapeados.
[control_12] -> 30 estímulos mapeados.
[control_13] -> 30 estímulos mapeados.
[control_35] -> 30 estímulos mapeados.
[control_36] -> 30 estímulos mapeados.
[control_37] -> 30 estímulos mapeados.
[control_38] -> 30 estímulos mapeados.
[control_39] -> 30 estímulos mapead

,Paciente_ID,Estimulo,Onset_Segundos
0,TEA_G_26,FF,1820.0
1,TEA_G_26,FN,1824.0
2,TEA_G_26,FR,1828.0
3,TEA_G_26,FF,1832.0
4,TEA_G_26,FN,1836.0
5,TEA_G_26,FR,1840.0
6,TEA_G_26,FF,1844.0
7,TEA_G_26,FN,1848.0
8,TEA_G_26,FR,1852.0
9,TEA_G_26,FF,1857.0


In [2]:
# EXTRAÇÃO DE ONSETS (TEMPOS EXATOS DOS ESTÍMULOS) COM AUDITORIA TEMPORAL

import os
import pandas as pd
import mne
import warnings

# Silenciar avisos para manter o terminal limpo
warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')

print("\n" + "="*80)
print("Iniciando Extração de Triggers (FF, FN, FR) com Cálculo de Delta T...")
print("="*80)

# 1. Parâmetros e Caminhos
DIR_RAW = '../data/raw/'
DIR_RESULTS = '../results/'

ARQUIVOS_EXCLUIDOS = ['control_14-sem marcacoes', 'TEA_M_22_1']
ESTIMULOS_ALVO = ['FF', 'FN', 'FR']

os.makedirs(DIR_RESULTS, exist_ok=True)

arquivos_edf = sorted([f for f in os.listdir(DIR_RAW) if f.lower().endswith('.edf')])
dados_onsets = []
pacientes_processados = 0

# 2. Varredura e Extração Lógica
for arquivo in arquivos_edf:
    paciente_id = arquivo.replace('.edf', '')
    
    if paciente_id in ARQUIVOS_EXCLUIDOS:
        continue
        
    caminho_completo = os.path.join(DIR_RAW, arquivo)
    
    try:
        raw = mne.io.read_raw_edf(caminho_completo, preload=False, encoding='latin1', verbose=False)
        
        contador_faces = 0
        ultimo_onset = None
        
        # Iterar sobre todas as anotações do exame temporalmente
        for ann in raw.annotations:
            descricao = ann['description'].strip().upper()
            
            # Se a descrição for FF, FN ou FR, capturamos as métricas
            if descricao in ESTIMULOS_ALVO:
                contador_faces += 1
                onset_atual = ann['onset']
                duracao = ann['duration']
                
                # Calcular a diferença de tempo desde o último estímulo
                if ultimo_onset is not None:
                    delta_t = onset_atual - ultimo_onset
                else:
                    delta_t = 0.0 # É o primeiro estímulo, logo não há intervalo anterior
                    
                dados_onsets.append({
                    'Paciente_ID': paciente_id,
                    'Face_Ordem': contador_faces,
                    'Estimulo': descricao,
                    'Onset_Segundos': round(onset_atual, 4),
                    'Duracao_Segundos': round(duracao, 4),
                    'Delta_T_Anterior_Segs': round(delta_t, 4)
                })
                
                # Atualizar a memória do último onset para o próximo ciclo
                ultimo_onset = onset_atual
                
        print(f"[{paciente_id}] -> {contador_faces} faces mapeadas.")
        pacientes_processados += 1
        
    except Exception as e:
        print(f"[ERRO] Falha ao processar {paciente_id}: {e}")

# 3. Consolidação e Exportação
df_onsets = pd.DataFrame(dados_onsets)

caminho_csv = os.path.join(DIR_RESULTS, 'tabela_tempos_estimulos_faces.csv')
df_onsets.to_csv(caminho_csv, index=False)

print("\n" + "="*80)
print("EXTRAÇÃO CONCLUÍDA!")
print(f"Total de pacientes processados: {pacientes_processados}")
print(f"Total de estímulos isolados: {len(df_onsets)}")
print(f"Ficheiro salvo em: {caminho_csv}")
print("="*80)

# Exibe as primeiras 10 linhas para conferência imediata
display(df_onsets.head(10))


Iniciando Extração de Triggers (FF, FN, FR) com Cálculo de Delta T...
[TEA_G_26] -> 30 faces mapeadas.
[TEA_G_28] -> 30 faces mapeadas.
[TEA_G_30] -> 30 faces mapeadas.
[TEA_G_40] -> 30 faces mapeadas.
[TEA_L_16] -> 0 faces mapeadas.
[TEA_L_17] -> 30 faces mapeadas.
[TEA_L_24] -> 30 faces mapeadas.
[TEA_L_25] -> 27 faces mapeadas.
[TEA_L_27] -> 31 faces mapeadas.
[TEA_L_29] -> 30 faces mapeadas.
[TEA_L_32] -> 30 faces mapeadas.
[TEA_L_46] -> 30 faces mapeadas.
[TEA_M_15] -> 30 faces mapeadas.
[TEA_M_18] -> 30 faces mapeadas.
[TEA_M_19] -> 30 faces mapeadas.
[TEA_M_20] -> 30 faces mapeadas.
[TEA_M_21] -> 30 faces mapeadas.
[TEA_M_22] -> 30 faces mapeadas.
[control_12] -> 30 faces mapeadas.
[control_13] -> 30 faces mapeadas.
[control_35] -> 30 faces mapeadas.
[control_36] -> 30 faces mapeadas.
[control_37] -> 30 faces mapeadas.
[control_38] -> 30 faces mapeadas.
[control_39] -> 30 faces mapeadas.
[control_41] -> 36 faces mapeadas.
[control_42] -> 30 faces mapeadas.
[control_43] -> 30 fa

,Paciente_ID,Face_Ordem,Estimulo,Onset_Segundos,Duracao_Segundos,Delta_T_Anterior_Segs
0,TEA_G_26,1,FF,1820.0,0.0,0.0
1,TEA_G_26,2,FN,1824.0,0.0,4.0
2,TEA_G_26,3,FR,1828.0,0.0,4.0
3,TEA_G_26,4,FF,1832.0,0.0,4.0
4,TEA_G_26,5,FN,1836.0,0.0,4.0
5,TEA_G_26,6,FR,1840.0,0.0,4.0
6,TEA_G_26,7,FF,1844.0,0.0,4.0
7,TEA_G_26,8,FN,1848.0,0.0,4.0
8,TEA_G_26,9,FR,1852.0,0.0,4.0
9,TEA_G_26,10,FF,1857.0,0.0,5.0


In [4]:
# Exibe as primeiras 10 linhas para conferência imediata
display(df_onsets.head(30))

,Paciente_ID,Face_Ordem,Estimulo,Onset_Segundos,Duracao_Segundos,Delta_T_Anterior_Segs
0,TEA_G_26,1,FF,1820.0,0.0,0.0
1,TEA_G_26,2,FN,1824.0,0.0,4.0
2,TEA_G_26,3,FR,1828.0,0.0,4.0
3,TEA_G_26,4,FF,1832.0,0.0,4.0
4,TEA_G_26,5,FN,1836.0,0.0,4.0
5,TEA_G_26,6,FR,1840.0,0.0,4.0
6,TEA_G_26,7,FF,1844.0,0.0,4.0
7,TEA_G_26,8,FN,1848.0,0.0,4.0
8,TEA_G_26,9,FR,1852.0,0.0,4.0
9,TEA_G_26,10,FF,1857.0,0.0,5.0


In [5]:
import os
import mne
import collections
import warnings

# Silenciar avisos para manter o terminal limpo
warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')

print("\n" + "="*80)
print("Iniciando Auditoria Forense de Marcadores nos Pacientes Anômalos...")
print("="*80)

# Parâmetros
DIR_RAW = '../data/raw/'
PACIENTES_AUDITORIA = ['TEA_L_16', 'TEA_L_25', 'TEA_L_27', 'control_41']

for paciente_id in PACIENTES_AUDITORIA:
    caminho_completo = os.path.join(DIR_RAW, f"{paciente_id}.edf")
    
    if not os.path.exists(caminho_completo):
        print(f"[AVISO] Arquivo não encontrado: {caminho_completo}")
        continue
        
    try:
        # Lê o ficheiro sem carregar os dados cerebrais para a memória (rápido)
        raw = mne.io.read_raw_edf(caminho_completo, preload=False, encoding='latin1', verbose=False)
        
        # Coleta absolutamente todas as descrições registadas
        todas_descricoes = [ann['description'].strip() for ann in raw.annotations]
        
        # Conta a frequência de cada marcador
        contagem = collections.Counter(todas_descricoes)
        
        print(f"\n--- Paciente: {paciente_id} ---")
        print(f"Total absoluto de eventos registados: {len(todas_descricoes)}")
        print("Detalhamento dos marcadores encontrados (Nome Exato -> Quantidade):")
        
        # Imprime do marcador que mais apareceu para o que menos apareceu
        for marcador, qtd in contagem.most_common():
            # Colocamos o marcador entre aspas simples para detetar espaços em branco acidentais
            print(f"  -> '{marcador}': {qtd} vezes")
            
    except Exception as e:
        print(f"[ERRO] Falha ao processar {paciente_id}: {e}")

print("\n" + "="*80)
print("AUDITORIA CONCLUÍDA.")
print("="*80)


Iniciando Auditoria Forense de Marcadores nos Pacientes Anômalos...

--- Paciente: TEA_L_16 ---
Total absoluto de eventos registados: 423
Detalhamento dos marcadores encontrados (Nome Exato -> Quantidade):
  -> 'Calibracao': 37 vezes
  -> 'TREM': 30 vezes
  -> 'FACE': 30 vezes
  -> 'MONTAGEM': 30 vezes
  -> 'VIDEO': 10 vezes
  -> 'PROJECAO': 10 vezes
  -> 'F': 10 vezes
  -> 'N': 10 vezes
  -> 'R': 10 vezes
  -> 'TREM GRANDE': 7 vezes
  -> 'A1+A2 OFF': 4 vezes
  -> 'IMP CHECK ON': 3 vezes
  -> 'IMP CHECK OFF': 3 vezes
  -> 'Segment: REC START AV EEG': 3 vezes
  -> 'Olhos abertos': 2 vezes
  -> 'Olhos Fechados': 2 vezes
  -> '+0.000000': 1 vezes
  -> 'Segment: REC START DB EEG': 1 vezes
  -> '+1.140000': 1 vezes
  -> '+3.920000': 1 vezes
  -> '+5.000000': 1 vezes
  -> '+7.000000': 1 vezes
  -> '+61.940000': 1 vezes
  -> '+63.000000': 1 vezes
  -> '+73.000000': 1 vezes
  -> '+168.160000': 1 vezes
  -> '+170.000000': 1 vezes
  -> '+173.000000': 1 vezes
  -> '+174.040000': 1 vezes
  -> '+2

In [6]:
import os
import pandas as pd
import mne
import warnings

warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')

print("\n" + "="*80)
print("Iniciando Extração Definitiva (Com Correções Forenses)...")
print("="*80)

DIR_RAW = '../data/raw/'
DIR_RESULTS = '../results/'

# Pacientes a excluir (os mesmos de antes)
ARQUIVOS_EXCLUIDOS = ['control_14-sem marcacoes', 'TEA_M_22_1']

# Expandimos o dicionário de estímulos para incluir o TEA_L_16 ('F', 'N', 'R')
ESTIMULOS_ALVO = ['FF', 'FN', 'FR', 'F', 'N', 'R']

os.makedirs(DIR_RESULTS, exist_ok=True)
arquivos_edf = sorted([f for f in os.listdir(DIR_RAW) if f.lower().endswith('.edf')])
dados_onsets = []
pacientes_processados = 0

for arquivo in arquivos_edf:
    paciente_id = arquivo.replace('.edf', '')
    if paciente_id in ARQUIVOS_EXCLUIDOS: continue
        
    caminho_completo = os.path.join(DIR_RAW, arquivo)
    
    try:
        raw = mne.io.read_raw_edf(caminho_completo, preload=False, encoding='latin1', verbose=False)
        contador_faces = 0
        ultimo_onset = None
        
        # REGRA ESPECIAL PARA TEA_L_25: Usar o marcador genérico 'FACE'
        if paciente_id == 'TEA_L_25':
            for ann in raw.annotations:
                descricao = ann['description'].strip().upper()
                if descricao == 'FACE':
                    contador_faces += 1
                    onset_atual = ann['onset']
                    duracao = ann['duration']
                    delta_t = onset_atual - ultimo_onset if ultimo_onset is not None else 0.0
                    
                    # Deduzindo a emoção pela ordem (Blocos de 10: FF, depois FN, depois FR)
                    if contador_faces <= 10: estimulo_deduzido = 'FF'
                    elif contador_faces <= 20: estimulo_deduzido = 'FN'
                    else: estimulo_deduzido = 'FR'

                    dados_onsets.append({
                        'Paciente_ID': paciente_id,
                        'Face_Ordem': contador_faces,
                        'Estimulo': estimulo_deduzido,
                        'Onset_Segundos': round(onset_atual, 4),
                        'Duracao_Segundos': round(duracao, 4),
                        'Delta_T_Anterior_Segs': round(delta_t, 4)
                    })
                    ultimo_onset = onset_atual
            print(f"[{paciente_id}] -> {contador_faces} faces recuperadas via 'FACE'.")
            pacientes_processados += 1
            continue # Pula para o próximo paciente
            
        # EXTRAÇÃO PADRÃO (Com filtro de duplo clique para TEA_L_27 e control_41)
        for ann in raw.annotations:
            descricao = ann['description'].strip().upper()
            
            if descricao in ESTIMULOS_ALVO:
                onset_atual = ann['onset']
                delta_t = onset_atual - ultimo_onset if ultimo_onset is not None else 0.0
                
                # FILTRO DE DUPLO CLIQUE: Se a diferença for menor que 2 segundos, é erro do investigador. Ignora.
                if ultimo_onset is not None and delta_t < 2.0:
                    continue
                    
                contador_faces += 1
                duracao = ann['duration']
                
                # Normalizando as letras do TEA_L_16 para o padrão oficial
                if descricao == 'F': descricao = 'FF'
                if descricao == 'N': descricao = 'FN'
                if descricao == 'R': descricao = 'FR'
                
                dados_onsets.append({
                    'Paciente_ID': paciente_id,
                    'Face_Ordem': contador_faces,
                    'Estimulo': descricao,
                    'Onset_Segundos': round(onset_atual, 4),
                    'Duracao_Segundos': round(duracao, 4),
                    'Delta_T_Anterior_Segs': round(delta_t, 4)
                })
                ultimo_onset = onset_atual
                
        # Se apesar da limpeza o arquivo der mais de 30, avisamos.
        if contador_faces > 30:
            print(f"[ALERTA] {paciente_id} ainda possui {contador_faces} faces mapeadas.")
        else:
            print(f"[{paciente_id}] -> {contador_faces} faces mapeadas.")
            
        pacientes_processados += 1
        
    except Exception as e:
        print(f"[ERRO] Falha ao processar {paciente_id}: {e}")

df_onsets = pd.DataFrame(dados_onsets)
caminho_csv = os.path.join(DIR_RESULTS, 'tabela_tempos_estimulos_faces_limpa.csv')
df_onsets.to_csv(caminho_csv, index=False)

print("\n" + "="*80)
print("EXTRAÇÃO DEFINITIVA CONCLUÍDA!")
print(f"Total de pacientes na matriz: {pacientes_processados}")
print(f"Arquivo Salvo: {caminho_csv}")
print("="*80)


Iniciando Extração Definitiva (Com Correções Forenses)...
[TEA_G_26] -> 30 faces mapeadas.
[TEA_G_28] -> 30 faces mapeadas.
[TEA_G_30] -> 30 faces mapeadas.
[TEA_G_40] -> 30 faces mapeadas.
[TEA_L_16] -> 30 faces mapeadas.
[TEA_L_17] -> 30 faces mapeadas.
[TEA_L_24] -> 30 faces mapeadas.
[TEA_L_25] -> 30 faces recuperadas via 'FACE'.
[ALERTA] TEA_L_27 ainda possui 31 faces mapeadas.
[TEA_L_29] -> 30 faces mapeadas.
[TEA_L_32] -> 30 faces mapeadas.
[TEA_L_46] -> 30 faces mapeadas.
[TEA_M_15] -> 30 faces mapeadas.
[TEA_M_18] -> 30 faces mapeadas.
[TEA_M_19] -> 30 faces mapeadas.
[TEA_M_20] -> 30 faces mapeadas.
[TEA_M_21] -> 30 faces mapeadas.
[TEA_M_22] -> 30 faces mapeadas.
[ALERTA] control_12 ainda possui 31 faces mapeadas.
[control_13] -> 30 faces mapeadas.
[control_35] -> 30 faces mapeadas.
[control_36] -> 30 faces mapeadas.
[control_37] -> 30 faces mapeadas.
[control_38] -> 30 faces mapeadas.
[control_39] -> 30 faces mapeadas.
[ALERTA] control_41 ainda possui 36 faces mapeadas.
[c

In [8]:
import os
import pandas as pd
import mne
import warnings

warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')

print("\n" + "="*80)
print("Iniciando Extração Rigorosa (Forçando Exatamente 30 Faces)...")
print("="*80)

DIR_RAW = '../data/raw/'
DIR_RESULTS = '../results/'

# Pacientes a excluir
ARQUIVOS_EXCLUIDOS = ['control_14-sem marcacoes', 'TEA_M_22_1']
ESTIMULOS_ALVO = ['FF', 'FN', 'FR']

os.makedirs(DIR_RESULTS, exist_ok=True)
arquivos_edf = sorted([f for f in os.listdir(DIR_RAW) if f.lower().endswith('.edf')])
dados_onsets = []
pacientes_processados = 0

for arquivo in arquivos_edf:
    paciente_id = arquivo.replace('.edf', '')
    if paciente_id in ARQUIVOS_EXCLUIDOS: continue
        
    caminho_completo = os.path.join(DIR_RAW, arquivo)
    
    try:
        raw = mne.io.read_raw_edf(caminho_completo, preload=False, encoding='latin1', verbose=False)
        
        eventos_paciente = []
        
        # 1. VARREDURA BRUTA (Captura tudo o que for válido)
        for ann in raw.annotations:
            descricao = ann['description'].strip().upper()
            onset = ann['onset']
            duracao = ann['duration']
            
            # Regra Restrita ao TEA_L_16: Traduzir letras soltas
            if paciente_id == 'TEA_L_16':
                if descricao == 'F': descricao = 'FF'
                elif descricao == 'N': descricao = 'FN'
                elif descricao == 'R': descricao = 'FR'
                
            # Regra Restrita ao TEA_L_25: Capturar o marcador genérico
            if paciente_id == 'TEA_L_25' and descricao == 'FACE':
                eventos_paciente.append({'estimulo': 'FACE', 'onset': onset, 'duracao': duracao})
                continue
                
            # Restante dos pacientes: Capturar apenas os alvos
            if descricao in ESTIMULOS_ALVO:
                eventos_paciente.append({'estimulo': descricao, 'onset': onset, 'duracao': duracao})
                
        # 2. FILTRAGEM CIRÚRGICA (Garantir 30 Faces Exatas)
        eventos_filtrados = []
        
        if paciente_id == 'TEA_L_25':
            # Pega exatamente os 30 primeiros blocos genéricos e deduz a emoção
            for i, ev in enumerate(eventos_paciente[:30]):
                est = 'FF' if i < 10 else ('FN' if i < 20 else 'FR')
                eventos_filtrados.append({'estimulo': est, 'onset': ev['onset'], 'duracao': ev['duracao']})
        else:
            # Separa por emoção
            ff_list = [ev for ev in eventos_paciente if ev['estimulo'] == 'FF']
            fn_list = [ev for ev in eventos_paciente if ev['estimulo'] == 'FN']
            fr_list = [ev for ev in eventos_paciente if ev['estimulo'] == 'FR']
            
            # A REGRA DOS ÚLTIMOS 10: Se houver mais de 10, mantém apenas os últimos 10 (as tentativas válidas finais)
            ff_list = ff_list[-10:] if len(ff_list) > 10 else ff_list
            fn_list = fn_list[-10:] if len(fn_list) > 10 else fn_list
            fr_list = fr_list[-10:] if len(fr_list) > 10 else fr_list
            
            # Reagrupa e ordena cronologicamente pelo onset
            eventos_filtrados = sorted(ff_list + fn_list + fr_list, key=lambda x: x['onset'])

        # 3. CÁLCULO DO DELTA T E MONTAGEM DA TABELA
        ultimo_onset = None
        for i, ev in enumerate(eventos_filtrados):
            onset_atual = ev['onset']
            delta_t = onset_atual - ultimo_onset if ultimo_onset is not None else 0.0
            
            dados_onsets.append({
                'Paciente_ID': paciente_id,
                'Face_Ordem': i + 1,
                'Estimulo': ev['estimulo'],
                'Onset_Segundos': round(onset_atual, 4),
                'Duracao_Segundos': round(ev['duracao'], 4),
                'Delta_T_Anterior_Segs': round(delta_t, 4)
            })
            ultimo_onset = onset_atual
            
        print(f"[{paciente_id}] -> {len(eventos_filtrados)} faces mapeadas e trancadas.")
        pacientes_processados += 1
        
    except Exception as e:
        print(f"[ERRO] Falha ao processar {paciente_id}: {e}")

df_onsets = pd.DataFrame(dados_onsets)
caminho_csv = os.path.join(DIR_RESULTS, 'tabela_tempos_estimulos_faces_limpa.csv')
df_onsets.to_csv(caminho_csv, index=False)

print("\n" + "="*80)
print("ALINHAMENTO CONCLUÍDO!")
print(f"Total de pacientes na matriz: {pacientes_processados}")
print(f"Arquivo Salvo: {caminho_csv}")
print("="*80)


Iniciando Extração Rigorosa (Forçando Exatamente 30 Faces)...
[TEA_G_26] -> 30 faces mapeadas e trancadas.
[TEA_G_28] -> 29 faces mapeadas e trancadas.
[TEA_G_30] -> 30 faces mapeadas e trancadas.
[TEA_G_40] -> 30 faces mapeadas e trancadas.
[TEA_L_16] -> 30 faces mapeadas e trancadas.
[TEA_L_17] -> 30 faces mapeadas e trancadas.
[TEA_L_24] -> 30 faces mapeadas e trancadas.
[TEA_L_25] -> 30 faces mapeadas e trancadas.
[TEA_L_27] -> 30 faces mapeadas e trancadas.
[TEA_L_29] -> 30 faces mapeadas e trancadas.
[TEA_L_32] -> 30 faces mapeadas e trancadas.
[TEA_L_46] -> 30 faces mapeadas e trancadas.
[TEA_M_15] -> 30 faces mapeadas e trancadas.
[TEA_M_18] -> 30 faces mapeadas e trancadas.
[TEA_M_19] -> 30 faces mapeadas e trancadas.
[TEA_M_20] -> 30 faces mapeadas e trancadas.
[TEA_M_21] -> 30 faces mapeadas e trancadas.
[TEA_M_22] -> 30 faces mapeadas e trancadas.
[control_12] -> 30 faces mapeadas e trancadas.
[control_13] -> 30 faces mapeadas e trancadas.
[control_35] -> 30 faces mapeadas

In [9]:
import os
import pandas as pd
import mne
import warnings

# Silenciar avisos para manter o terminal limpo
warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')

print("\n" + "="*80)
print("ALINHAMENTO FINAL: EXTRAÇÃO RIGOROSA (FOCO NO 30 PERFEITO)")
print("="*80)

DIR_RAW = '../data/raw/'
DIR_RESULTS = '../results/'

# Pacientes a excluir (sem marcações ou corrompidos)
ARQUIVOS_EXCLUIDOS = ['control_14-sem marcacoes', 'TEA_M_22_1']

os.makedirs(DIR_RESULTS, exist_ok=True)
arquivos_edf = sorted([f for f in os.listdir(DIR_RAW) if f.lower().endswith('.edf')])
dados_onsets = []
pacientes_processados = 0

for arquivo in arquivos_edf:
    paciente_id = arquivo.replace('.edf', '')
    if paciente_id in ARQUIVOS_EXCLUIDOS: continue
        
    caminho_completo = os.path.join(DIR_RAW, arquivo)
    
    try:
        raw = mne.io.read_raw_edf(caminho_completo, preload=False, encoding='latin1', verbose=False)
        eventos_raw = []
        
        # 1. VARREDURA INDIVIDUALIZADA
        for ann in raw.annotations:
            desc = ann['description'].strip().upper()
            onset = ann['onset']
            dur = ann['duration']
            
            # --- REGRA PARA TEA_L_16 (Mapear letras soltas) ---
            if paciente_id == 'TEA_L_16':
                if desc == 'F': desc = 'FF'
                elif desc == 'N': desc = 'FN'
                elif desc == 'R': desc = 'FR'
            
            # --- REGRA PARA TEA_L_25 (Mapear marcador genérico) ---
            if paciente_id == 'TEA_L_25' and desc == 'FACE':
                eventos_raw.append({'estimulo': 'FACE', 'onset': onset, 'duracao': dur})
                continue
                
            # --- REGRA PADRÃO (Para todos os outros, incluindo TEA_G_28) ---
            if desc in ['FF', 'FN', 'FR']:
                eventos_raw.append({'estimulo': desc, 'onset': onset, 'duracao': dur})
                
        # 2. FILTRAGEM E CATEGORIZAÇÃO
        eventos_finais = []
        
        if paciente_id == 'TEA_L_25':
            # Recupera as 30 primeiras e rotula por ordem
            for i, ev in enumerate(eventos_raw[:30]):
                label = 'FF' if i < 10 else ('FN' if i < 20 else 'FR')
                eventos_finais.append({'estimulo': label, 'onset': ev['onset'], 'duracao': ev['duracao']})
        else:
            # Separa por emoção e aplica a REGRA DOS ÚLTIMOS 10 (para limpar control_41 e TEA_L_27)
            felizes = [e for e in eventos_raw if e['estimulo'] == 'FF'][-10:]
            neutras = [e for e in eventos_raw if e['estimulo'] == 'FN'][-10:]
            raiva   = [e for e in eventos_raw if e['estimulo'] == 'FR'][-10:]
            
            # Reorganiza na ordem do tempo
            eventos_finais = sorted(felizes + neutras + raiva, key=lambda x: x['onset'])

        # 3. VALIDAÇÃO E MONTAGEM DO CSV
        if len(eventos_finais) != 30:
            print(f"[ALERTA] {paciente_id} ficou com {len(eventos_finais)} faces. (FF:{len(felizes)} | FN:{len(neutras)} | FR:{len(raiva)})")
        else:
            print(f"[{paciente_id}] -> {len(eventos_finais)} faces OK.")
            
        ultimo_onset = None
        for i, ev in enumerate(eventos_finais):
            onset_atual = ev['onset']
            delta_t = onset_atual - ultimo_onset if ultimo_onset is not None else 0.0
            
            dados_onsets.append({
                'Paciente_ID': paciente_id,
                'Face_Ordem': i + 1,
                'Estimulo': ev['estimulo'],
                'Onset_Segundos': round(onset_atual, 4),
                'Duracao_Segundos': round(ev['duracao'], 4),
                'Delta_T_Anterior_Segs': round(delta_t, 4)
            })
            ultimo_onset = onset_atual
            
        pacientes_processados += 1
        
    except Exception as e:
        print(f"[ERRO] Falha ao processar {paciente_id}: {e}")

# 4. SALVAMENTO
df_final = pd.DataFrame(dados_onsets)
caminho_csv = os.path.join(DIR_RESULTS, 'tabela_tempos_estimulos_faces_limpa.csv')
df_final.to_csv(caminho_csv, index=False)

print("\n" + "="*80)
print(f"CONCLUÍDO: {pacientes_processados} pacientes processados.")
print(f"Matriz salva em: {caminho_csv}")
print("="*80)


ALINHAMENTO FINAL: EXTRAÇÃO RIGOROSA (FOCO NO 30 PERFEITO)
[TEA_G_26] -> 30 faces OK.
[ALERTA] TEA_G_28 ficou com 29 faces. (FF:10 | FN:10 | FR:9)
[TEA_G_30] -> 30 faces OK.
[TEA_G_40] -> 30 faces OK.
[TEA_L_16] -> 30 faces OK.
[TEA_L_17] -> 30 faces OK.
[TEA_L_24] -> 30 faces OK.
[TEA_L_25] -> 30 faces OK.
[TEA_L_27] -> 30 faces OK.
[TEA_L_29] -> 30 faces OK.
[TEA_L_32] -> 30 faces OK.
[TEA_L_46] -> 30 faces OK.
[TEA_M_15] -> 30 faces OK.
[TEA_M_18] -> 30 faces OK.
[TEA_M_19] -> 30 faces OK.
[TEA_M_20] -> 30 faces OK.
[TEA_M_21] -> 30 faces OK.
[TEA_M_22] -> 30 faces OK.
[control_12] -> 30 faces OK.
[control_13] -> 30 faces OK.
[control_35] -> 30 faces OK.
[control_36] -> 30 faces OK.
[control_37] -> 30 faces OK.
[control_38] -> 30 faces OK.
[control_39] -> 30 faces OK.
[control_41] -> 30 faces OK.
[control_42] -> 30 faces OK.
[control_43] -> 30 faces OK.
[control_44] -> 30 faces OK.
[control_47] -> 30 faces OK.
[control_48] -> 30 faces OK.
[control_49] -> 30 faces OK.
[control_50] -

In [10]:
import os
import pandas as pd
import mne
import warnings

warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')

print("\n" + "="*80)
print("ALINHAMENTO FINAL: EXTRAÇÃO RIGOROSA E INTELIGENTE")
print("="*80)

DIR_RAW = '../data/raw/'
DIR_RESULTS = '../results/'

ARQUIVOS_EXCLUIDOS = ['control_14-sem marcacoes', 'TEA_M_22_1']

os.makedirs(DIR_RESULTS, exist_ok=True)
arquivos_edf = sorted([f for f in os.listdir(DIR_RAW) if f.lower().endswith('.edf')])
dados_onsets = []
pacientes_processados = 0

for arquivo in arquivos_edf:
    paciente_id = arquivo.replace('.edf', '')
    if paciente_id in ARQUIVOS_EXCLUIDOS: continue
        
    caminho_completo = os.path.join(DIR_RAW, arquivo)
    
    try:
        raw = mne.io.read_raw_edf(caminho_completo, preload=False, encoding='latin1', verbose=False)
        eventos_raw = []
        
        # 1. VARREDURA 
        for ann in raw.annotations:
            desc = ann['description'].strip().upper()
            onset = ann['onset']
            dur = ann['duration']
            
            if paciente_id == 'TEA_L_16':
                if desc == 'F': desc = 'FF'
                elif desc == 'N': desc = 'FN'
                elif desc == 'R': desc = 'FR'
            
            if paciente_id == 'TEA_L_25' and desc == 'FACE':
                eventos_raw.append({'estimulo': 'FACE', 'onset': onset, 'duracao': dur})
                continue
                
            if desc in ['FF', 'FN', 'FR']:
                eventos_raw.append({'estimulo': desc, 'onset': onset, 'duracao': dur})
                
        # 2. FILTRAGEM INTELIGENTE
        eventos_finais = []
        
        if paciente_id == 'TEA_L_25':
            for i, ev in enumerate(eventos_raw[:30]):
                label = 'FF' if i < 10 else ('FN' if i < 20 else 'FR')
                eventos_finais.append({'estimulo': label, 'onset': ev['onset'], 'duracao': ev['duracao']})
        else:
            # SE HOUVER EXCESSO (> 30), aplica o filtro de limpeza dos últimos 10
            if len(eventos_raw) > 30:
                felizes = [e for e in eventos_raw if e['estimulo'] == 'FF'][-10:]
                neutras = [e for e in eventos_raw if e['estimulo'] == 'FN'][-10:]
                raiva   = [e for e in eventos_raw if e['estimulo'] == 'FR'][-10:]
                eventos_finais = sorted(felizes + neutras + raiva, key=lambda x: x['onset'])
            # SE TIVER 30 OU MENOS, preserva a cronologia absoluta do equipamento
            else:
                eventos_finais = sorted(eventos_raw, key=lambda x: x['onset'])

        # 3. VALIDAÇÃO E MONTAGEM
        if len(eventos_finais) != 30:
            print(f"[ALERTA] {paciente_id} tem {len(eventos_finais)} faces.")
        else:
            print(f"[{paciente_id}] -> {len(eventos_finais)} faces OK.")
            
        ultimo_onset = None
        for i, ev in enumerate(eventos_finais):
            onset_atual = ev['onset']
            delta_t = onset_atual - ultimo_onset if ultimo_onset is not None else 0.0
            
            dados_onsets.append({
                'Paciente_ID': paciente_id,
                'Face_Ordem': i + 1,
                'Estimulo': ev['estimulo'],
                'Onset_Segundos': round(onset_atual, 4),
                'Duracao_Segundos': round(ev['duracao'], 4),
                'Delta_T_Anterior_Segs': round(delta_t, 4)
            })
            ultimo_onset = onset_atual
            
        pacientes_processados += 1
        
    except Exception as e:
        print(f"[ERRO] Falha ao processar {paciente_id}: {e}")

df_final = pd.DataFrame(dados_onsets)
caminho_csv = os.path.join(DIR_RESULTS, 'tabela_tempos_estimulos_faces_limpa.csv')
df_final.to_csv(caminho_csv, index=False)

print("\n" + "="*80)
print(f"CONCLUÍDO! Matriz salva em: {caminho_csv}")
print("="*80)


ALINHAMENTO FINAL: EXTRAÇÃO RIGOROSA E INTELIGENTE
[TEA_G_26] -> 30 faces OK.
[TEA_G_28] -> 30 faces OK.
[TEA_G_30] -> 30 faces OK.
[TEA_G_40] -> 30 faces OK.
[TEA_L_16] -> 30 faces OK.
[TEA_L_17] -> 30 faces OK.
[TEA_L_24] -> 30 faces OK.
[TEA_L_25] -> 30 faces OK.
[TEA_L_27] -> 30 faces OK.
[TEA_L_29] -> 30 faces OK.
[TEA_L_32] -> 30 faces OK.
[TEA_L_46] -> 30 faces OK.
[TEA_M_15] -> 30 faces OK.
[TEA_M_18] -> 30 faces OK.
[TEA_M_19] -> 30 faces OK.
[TEA_M_20] -> 30 faces OK.
[TEA_M_21] -> 30 faces OK.
[TEA_M_22] -> 30 faces OK.
[control_12] -> 30 faces OK.
[control_13] -> 30 faces OK.
[control_35] -> 30 faces OK.
[control_36] -> 30 faces OK.
[control_37] -> 30 faces OK.
[control_38] -> 30 faces OK.
[control_39] -> 30 faces OK.
[control_41] -> 30 faces OK.
[control_42] -> 30 faces OK.
[control_43] -> 30 faces OK.
[control_44] -> 30 faces OK.
[control_47] -> 30 faces OK.
[control_48] -> 30 faces OK.
[control_49] -> 30 faces OK.
[control_50] -> 30 faces OK.
[control_51] -> 30 faces OK